In [1]:
import pandas as pd
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
from transformers import get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import ast
from sklearn.metrics import f1_score, classification_report
from torch.amp import autocast, GradScaler
import random
import os
import numpy as np


In [2]:

def set_seed(seed=42):
    """Locks all random number generators for exact reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)


set_seed(42)


### Dataset generation

In [3]:
# Load all the data 

# Column names for binary classification PCL data
column_names = ['par_id', 'art_id', 'keyword', 'country_code', 'text', 'label']

# Reading the data, skipping the first 4 lines of disclaimer
# delimiter is set to \t for tab-separated values
df_pcl = pd.read_csv('data/dontpatronizeme_pcl.tsv', 
                 sep='\t', 
                 skiprows=4, 
                 names=column_names, 
                 index_col=False,
                 quoting=3)


# data load for multi label classification

span_columns = [
    'par_id', 'art_id', 'text', 'keyword', 'country_code', 
    'span_start', 'span_finish', 'span_text', 'pcl_category', 'num_annotators'
]

# Load the data
df_categories= pd.read_csv('data/dontpatronizeme_categories.tsv', 
                       sep='\t', 
                       skiprows=4, 
                       names=span_columns, 
                       index_col=False,
                       quoting=3) # quoting=3 tells pandas to ignore quotes to avoid splitting text mid-sentence

# Training labels
df_train_labels = pd.read_csv('data/train_semeval_parids-labels.csv', index_col=False)

df_dev_labels = pd.read_csv('data/dev_semeval_parids-labels.csv', index_col=False)
                             


In [4]:
# Binary PCL classifcation
# Create the new binary column 'pcl_presence'
df_pcl['pcl_presence'] = df_pcl['label'].apply(lambda x: 0 if x in [0, 1] else 1)

import ast
df_train_labels['label'] = df_train_labels['label'].apply(
        lambda x: ast.literal_eval(x) if isinstance(x, str) else x
    )

df_dev_labels['label'] = df_dev_labels['label'].apply(
        lambda x: ast.literal_eval(x) if isinstance(x, str) else x
    )

In [5]:
import html
import re

# Clean data 

def clean_pcl_text(text):
    text = str(text)
    
    # Convert HTML entities back to normal characters (e.g., &amp; -> &)
    text = html.unescape(text)
    
    # Strip out remaining HTML tags (e.g., <br>, <i>)
    text = re.sub(r'<[^>]+>', ' ', text)
    
    # 3Fix multiple spaces that might have been created by removing tags
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

# Apply the cleaning function to the dataset

df_pcl['text'] = df_pcl['text'].apply(clean_pcl_text)

# Verify the cleaning worked
html_pattern = r'<[^>]+>'
entity_pattern = r'&[a-z]+;'

remaining_html = df_pcl['text'].str.contains(html_pattern, regex=True).sum()
remaining_entities = df_pcl['text'].str.contains(entity_pattern, regex=True).sum()

print(f"Remaining HTML tags and entities:{remaining_html} , {remaining_entities}")


Remaining HTML tags and entities:0 , 0


In [6]:
# merge train and dev labels with pcl presence info

import pandas as pd
import ast


def prepare_final_dataset(df_labels, df_main_text):
    
    # Merge to get the text, keyword, and country code based on par_id
    df_merged = pd.merge(df_labels, df_main_text[['par_id', 'text', 'keyword', 'country_code',"pcl_presence"]], 
                         on='par_id', how='left')
    
    # Format the input text (Keyword + Country + Text)
    # Using RoBERTa/DeBERTa's separator token </s> 
    df_merged['model_input'] = (
        # df_merged['keyword'].astype(str) + " </s> " + 
        # df_merged['country_code'].astype(str) + " </s> " + 
        df_merged['text'].astype(str)
    )
        
    return df_merged[['par_id', 'model_input', 'pcl_presence', 'label']]


train_val_data = prepare_final_dataset(df_train_labels, df_pcl)
dev_data = prepare_final_dataset(df_dev_labels, df_pcl)


# Split train data into train and validate
train_data, val_data = train_test_split(
    train_val_data, 
    test_size=0.15, # 15% goes to validation
    stratify=train_val_data['pcl_presence'], 
    random_state=42 # 
)

train_data = train_data.reset_index(drop=True)
val_data = val_data.reset_index(drop=True)

print(train_data.head(3))

   par_id                                        model_input  pcl_presence  \
0    1748  Contrary to such measures , in Sri Lanka , the...             0   
1    3205  "He said the city has become unaffordable for ...             0   
2    4237  "On the same page as the excellent letter from...             0   

                   label  
0  [0, 0, 0, 0, 0, 0, 0]  
1  [0, 0, 0, 0, 0, 0, 0]  
2  [0, 0, 0, 0, 0, 0, 0]  


In [7]:
model_name = 'roberta-base'


class PCLMultiTaskDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=256):
        self.texts = df['model_input'].tolist()
        self.binary_labels = df['pcl_presence'].tolist()
        # self.multi_labels = df['label'].tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        encoding = self.tokenizer(
            text, add_special_tokens=True, max_length=self.max_length,
            padding='max_length', truncation=True, return_attention_mask=True, return_tensors='pt'
        )
        
        binary_label = torch.tensor([self.binary_labels[idx]], dtype=torch.float32)
        
        # m_label = self.multi_labels[idx]
        # if isinstance(m_label, str):
        #     m_label = ast.literal_eval(m_label)
        # multi_label_tensor = torch.tensor(m_label, dtype=torch.float32)

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'binary_labels': binary_label,
            # 'multi_labels': multi_label_tensor
        }



In [8]:
# Initialize Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Build Datasets
train_dataset = PCLMultiTaskDataset(train_data, tokenizer)
val_dataset = PCLMultiTaskDataset(val_data, tokenizer)
dev_dataset = PCLMultiTaskDataset(dev_data, tokenizer)

# Build DataLoaders
train_loader = DataLoader(train_dataset, batch_size=8, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=8,shuffle=False )
dev_loader = DataLoader(dev_dataset, batch_size=8, shuffle=False)

### Model Specs

In [9]:
#remove leftover cache from previous runs
torch.cuda.empty_cache()

# Setup Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")


Device: cuda


In [10]:
import torch
import numpy as np
from sklearn.metrics import f1_score, classification_report
from transformers import AutoModelForSequenceClassification

print("Initializing Pre-trained RoBERTa")

baseline_model = AutoModelForSequenceClassification.from_pretrained(
    "roberta-base", 
    num_labels=1 # Binary classification
).to(device)

baseline_model.eval() 


val_probs, val_labels = [], []

with torch.no_grad():
    for batch in val_loader:
        bin_logits = baseline_model(
            batch['input_ids'].to(device), 
            attention_mask=batch['attention_mask'].to(device)
        ).logits
        
        probs = torch.sigmoid(bin_logits)
        val_probs.extend(probs.cpu().numpy())
        val_labels.extend(batch['binary_labels'].numpy())
        
y_val_true = np.array(val_labels).flatten()
y_val_probs = np.array(val_probs).flatten()

best_untrained_f1, best_untrained_thresh = 0.0, 0.5
for thresh in np.arange(0.1, 0.9, 0.01):
    preds = (y_val_probs >= thresh).astype(int)
    current_f1 = f1_score(y_val_true, preds, pos_label=1, zero_division=0)
    if current_f1 > best_untrained_f1:
        best_untrained_f1, best_untrained_thresh = current_f1, thresh
        
print(f"Untrained Peak Val F1: {best_untrained_f1:.4f} (at threshold {best_untrained_thresh:.2f})")

# ==========================================
# --- PHASE 2: UNTRAINED DEV EVALUATION ---
# ==========================================
print("\nRunning Untrained Model on Unseen Dev Set...")
dev_probs, dev_labels = [], []

with torch.no_grad():
    for batch in dev_loader:
        bin_logits = baseline_model(
            batch['input_ids'].to(device), 
            attention_mask=batch['attention_mask'].to(device)
        ).logits
        
        probs = torch.sigmoid(bin_logits) 
        dev_probs.extend(probs.cpu().numpy())
        dev_labels.extend(batch['binary_labels'].numpy())
        
y_dev_true = np.array(dev_labels).flatten()
y_dev_probs = np.array(dev_probs).flatten()

final_dev_preds = (y_dev_probs >= best_untrained_thresh).astype(int)
final_untrained_f1 = f1_score(y_dev_true, final_dev_preds, pos_label=1, zero_division=0)

print(f"\n UNTRAINED (ZERO-SHOT) DEV F1-SCORE: {final_untrained_f1:.4f}")
print("\n--- FINAL CLASSIFICATION REPORT ---")
print(classification_report(y_dev_true, final_dev_preds, target_names=['Non-PCL (0)', 'PCL (1)'], zero_division=0))

Initializing Pre-trained RoBERTa


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Untrained Peak Val F1: 0.1730 (at threshold 0.10)

Running Untrained Model on Unseen Dev Set...

 UNTRAINED (ZERO-SHOT) DEV F1-SCORE: 0.1736

--- FINAL CLASSIFICATION REPORT ---
              precision    recall  f1-score   support

 Non-PCL (0)       0.00      0.00      0.00      1895
     PCL (1)       0.10      1.00      0.17       199

    accuracy                           0.10      2094
   macro avg       0.05      0.50      0.09      2094
weighted avg       0.01      0.10      0.02      2094



### Training and validation

In [11]:
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import f1_score, classification_report
from torch.amp import autocast, GradScaler

# ==========================================
# 1. THE VANILLA DATASET (Raw Text Only)
# ==========================================
class VanillaPCLDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=256):
        # CRITICAL: We only use the raw 'text' here, completely ignoring the metadata
        self.texts = df['text'].tolist()
        self.labels = df['pcl_presence'].tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        encoding = self.tokenizer(
            text, add_special_tokens=True, max_length=self.max_length,
            padding='max_length', truncation=True, return_attention_mask=True, return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'binary_labels': torch.tensor([self.labels[idx]], dtype=torch.float32)
        }

# ==========================================
# 2. STANDARD DATALOADERS (No Class Balancing)
# ==========================================
print("📦 Building Vanilla DataLoaders...")
tokenizer = AutoTokenizer.from_pretrained("roberta-base")

train_dataset_vanilla = VanillaPCLDataset(train_data, tokenizer)
val_dataset_vanilla = VanillaPCLDataset(val_data, tokenizer)
dev_dataset_vanilla = VanillaPCLDataset(dev_data, tokenizer)

# CRITICAL: Notice there is NO sampler here. We just shuffle the imbalanced data.
train_loader_vanilla = DataLoader(train_dataset_vanilla, batch_size=8, shuffle=True, drop_last=True)
val_loader_vanilla = DataLoader(val_dataset_vanilla, batch_size=8, shuffle=False)
dev_loader_vanilla = DataLoader(dev_dataset_vanilla, batch_size=8, shuffle=False)

# ==========================================
# 3. VANILLA MODEL & OPTIMIZER
# ==========================================
print("🧠 Initializing Standard Sequence Classification Model...")
baseline_model = AutoModelForSequenceClassification.from_pretrained(
    "roberta-base", num_labels=1
).to(device)

optimizer = torch.optim.AdamW(baseline_model.parameters(), lr=1e-5)
criterion = nn.BCEWithLogitsLoss()
scaler = GradScaler("cuda")

# ==========================================
# 4. TRAINING & EVALUATION LOOP
# ==========================================
EPOCHS = 3
best_vanilla_val_f1 = 0.0
locked_vanilla_thresh = 0.5

for epoch in range(EPOCHS):
    baseline_model.train()
    total_loss = 0
    print(f"\n🚀 --- Starting Vanilla Epoch {epoch + 1}/{EPOCHS} ---")
    
    for step, batch in enumerate(train_loader_vanilla):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        bin_labels = batch['binary_labels'].to(device)
        
        optimizer.zero_grad()
        
        with autocast(device_type="cuda", dtype=torch.float16):
            # The official HuggingFace model returns an object where .logits has the predictions
            bin_logits = baseline_model(input_ids, attention_mask=attention_mask).logits
            loss = criterion(bin_logits, bin_labels)
            
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(baseline_model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        
        total_loss += loss.item()
            
    print(f"✅ Epoch {epoch + 1} Train Loss: {total_loss / len(train_loader_vanilla):.4f}")
    
    # --- PHASE 1: VALIDATION SWEEP ---
    baseline_model.eval() 
    val_probs, val_labels = [], []
    
    with torch.no_grad():
        for batch in val_loader_vanilla:
            bin_logits = baseline_model(
                batch['input_ids'].to(device), attention_mask=batch['attention_mask'].to(device)
            ).logits
            probs = torch.sigmoid(bin_logits) 
            val_probs.extend(probs.cpu().numpy())
            val_labels.extend(batch['binary_labels'].numpy())
            
    y_val_true = np.array(val_labels).flatten()
    y_val_probs = np.array(val_probs).flatten()
    
    epoch_best_f1, epoch_best_thresh = 0.0, 0.5
    for thresh in np.arange(0.1, 0.9, 0.01):
        preds = (y_val_probs >= thresh).astype(int)
        current_f1 = f1_score(y_val_true, preds, pos_label=1, zero_division=0)
        if current_f1 > epoch_best_f1:
            epoch_best_f1, epoch_best_thresh = current_f1, thresh
            
    print(f"🎯 Epoch {epoch + 1} Peak Val F1: {epoch_best_f1:.4f} (at threshold {epoch_best_thresh:.2f})")
    
    if epoch_best_f1 > best_vanilla_val_f1:
        best_vanilla_val_f1 = epoch_best_f1
        locked_vanilla_thresh = epoch_best_thresh
        # Saving strictly to RAM for the baseline to save disk space
        best_vanilla_weights = {k: v.cpu() for k, v in baseline_model.state_dict().items()}

# --- PHASE 2: UNSEEN DEV EVALUATION ---
print("\n==============================================")
print("🔄 Loading Best Vanilla Model for Final Dev Evaluation...")
baseline_model.load_state_dict({k: v.to(device) for k, v in best_vanilla_weights.items()})
baseline_model.eval()

dev_probs, dev_labels = [], []

with torch.no_grad():
    for batch in dev_loader_vanilla:
        bin_logits = baseline_model(
            batch['input_ids'].to(device), attention_mask=batch['attention_mask'].to(device)
        ).logits
        probs = torch.sigmoid(bin_logits) 
        dev_probs.extend(probs.cpu().numpy())
        dev_labels.extend(batch['binary_labels'].numpy())
        
y_dev_true = np.array(dev_labels).flatten()
y_dev_probs = np.array(dev_probs).flatten()

final_dev_preds = (y_dev_probs >= locked_vanilla_thresh).astype(int)
final_dev_f1 = f1_score(y_dev_true, final_dev_preds, pos_label=1, zero_division=0)

print(f"\n🏆 TRUE VANILLA DEV F1-SCORE: {final_dev_f1:.4f}")
print("\n--- FINAL CLASSIFICATION REPORT ---")
print(classification_report(y_dev_true, final_dev_preds, target_names=['Non-PCL (0)', 'PCL (1)'], zero_division=0))

📦 Building Vanilla DataLoaders...


KeyError: 'text'